In [443]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split

*шаблоны для s1, s2, t1, t2*

In [444]:
# базовый шаблон
class st_net(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), #
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], output_dim),
            nn.Tanh()
        )

    def forward(self, input):
        return self.net(input)

# линейное преобразование
class LinearNet(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), #
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.Linear(hidden_dims[1], output_dim),
        )

    def forward(self, input):
        return self.net(input)

# линейное преобразование
class SimpleLinearNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, output_dim),
        )

    def forward(self, input):
        return self.net(input)



In [445]:
# собственно, сама сеть
class INN(nn.Module):
    def __init__(self, input_dim1, input_dim2,
                hidden_dims_s1 = [64, 64], hidden_dims_s2 = [64, 64],
                hidden_dims_t1 = [64, 64], hidden_dims_t2 = [64, 64]):

        super().__init__()
        self.input = None
        self.input_dim1 = input_dim1
        self.input_dim2 = input_dim2

        self.u1 = None
        self.u2 = None
        self.v1 = None
        self.v2 = None

        self.s1_net = st_net(input_dim2, hidden_dims_s1, input_dim1)
        self.s2_net = st_net(input_dim1, hidden_dims_s2, input_dim2)
        self.t1_net = st_net(input_dim2, hidden_dims_t1, input_dim1)
        self.t2_net = st_net(input_dim1, hidden_dims_t2, input_dim2)


    def forward(self, input, reverse=False):
        if not reverse:  # Прямое преобразование

            self.input = input
            self.u1 = self.input[:, :self.input_dim1]
            self.u2 = self.input[:, self.input_dim1:]

            s1 = self.s1_net(self.u2)
            t1 = self.t1_net(self.u2)
            v1 = self.u1 * s1 + t1

            s2 = self.s2_net(v1)
            t2 = self.t2_net(v1)
            v2 = self.u2 * s2 + t2

            self.v1 = v1
            self.v2 = v2
            output = torch.cat((v1, v2), dim=1)

            return output

        else: # Обратный проход

            s2 = self.s2_net(self.v1)
            t2 = self.t2_net(self.v1)
            u2 = (self.v2 - t2) / (s2)

            s1 = self.s1_net(u2)
            t1 = self.t1_net(u2)
            u1 = (self.v1 - t1) / (s1)

            rerv_input = torch.cat((u1, u2), dim=1)

            return rerv_input

Проверим, что сеть "обращается"

In [446]:
def test_invertible_network():
    batch_size = 3
    model = INN(input_dim1=3, input_dim2=7)
    x = torch.randn(batch_size, 10)
    z = model(x)
    x_rev = model(z, reverse=True)
    print(f" x_fwd ={x}, \n x_rev ={x_rev}, \n z = {z}")
    reconstruction_error = F.mse_loss(x, x_rev)

    return model, reconstruction_error

In [447]:
model, reconstruction_error = test_invertible_network()
print(f"Reconstruction error: {reconstruction_error}")


 x_fwd =tensor([[-2.9080,  1.8429, -0.7274,  2.1532,  1.4176,  0.2212,  2.2385,  0.0034,
         -0.0454, -1.1978],
        [ 0.8723,  0.7831,  1.2196,  1.4470,  1.0825,  0.8455,  0.6403,  0.3144,
         -0.0205,  0.9900],
        [ 0.1200, -1.1575, -1.0483, -0.5833, -0.9554, -0.4162, -0.0181, -0.0281,
          0.4145,  0.3182]]), 
 x_rev =tensor([[-2.9080,  1.8429, -0.7274,  2.1532,  1.4176,  0.2212,  2.2385,  0.0034,
         -0.0454, -1.1978],
        [ 0.8723,  0.7831,  1.2196,  1.4470,  1.0825,  0.8455,  0.6403,  0.3144,
         -0.0205,  0.9900],
        [ 0.1200, -1.1575, -1.0482, -0.5833, -0.9554, -0.4162, -0.0181, -0.0281,
          0.4145,  0.3182]], grad_fn=<CatBackward0>), 
 z = tensor([[-1.0931, -0.1657, -0.2359,  0.1509, -0.4271, -0.0405, -0.3498,  0.0626,
         -0.0031,  0.0268],
        [-0.0369,  0.0849,  0.0020,  0.0150, -0.2622, -0.0573, -0.2070,  0.0756,
         -0.0166,  0.1202],
        [-0.2068, -0.2082, -0.1090, -0.1287,  0.0736,  0.0105, -0.1065,  0.07

In [448]:
print(model)

INN(
  (s1_net): st_net(
    (net): Sequential(
      (0): Linear(in_features=7, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=3, bias=True)
      (5): Tanh()
    )
  )
  (s2_net): st_net(
    (net): Sequential(
      (0): Linear(in_features=3, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=7, bias=True)
      (5): Tanh()
    )
  )
  (t1_net): st_net(
    (net): Sequential(
      (0): Linear(in_features=7, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=3, bias=True)
      (5): Tanh()
    )
  )
  (t2_net): st_net(
    (net): Sequential(
      (0): Linear(in_features=3, out_features=64, bias=True)
      (1): ReLU()


# Тест на `"Искуственных данных"`

In [449]:
# Полностью аддитивная линейная сеть (без деления)
class SimpleLinearINN(nn.Module):
    def __init__(self, input_dim1, input_dim2):

        super().__init__()
        self.input_dim1 = input_dim1
        self.input_dim2 = input_dim2

        # Нам нужны только t_net для сдвига
        self.t1_net = SimpleLinearNet(input_dim2, input_dim1)
        self.t2_net = SimpleLinearNet(input_dim1, input_dim2)

    def forward(self, input, reverse=False):
        if not reverse:  # Прямой проход: v = u + t
            u1 = input[:, :self.input_dim1]
            u2 = input[:, self.input_dim1:]

            v1 = u1 + self.t1_net(u2)
            v2 = u2 + self.t2_net(v1)

            return torch.cat((v1, v2), dim=1)

        else: # Обратный проход: u = v - t
            v1 = input[:, :self.input_dim1]
            v2 = input[:, self.input_dim1:]

            u2 = v2 - self.t2_net(v1)
            u1 = v1 - self.t1_net(u2)

            return torch.cat((u1, u2), dim=1)

In [450]:
class SimpleDataset(torch.utils.data.Dataset): #clearing noise example
    def __init__(self, data, a, b):
        self.input = data
        self.labels = a * data + torch.randn(data.shape) * 0.1  + b #Trying to "predict" noise to clear it in future
        self.targets  = a * data  + b

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.targets[idx], self.labels[idx]

    def show_input(self):
        return self.input

In [451]:
line = torch.arange(-100, 100, 0.01)
data_rows = []
for _ in range(10):
    shuffled_indices = torch.randperm(line.shape[0])
    data_rows.append(line[shuffled_indices])
Data = torch.stack(data_rows)
Data = Data.T
Data

tensor([[-27.6400, -39.4300, -64.9000,  ..., -19.4200, -29.7900, -54.9500],
        [-79.5400, -28.0200, -60.3100,  ..., -19.4500, -55.1800,  25.0200],
        [ 13.6300,  19.8600,   6.7100,  ...,  27.3500,  49.7700, -48.8600],
        ...,
        [ 99.5200, -87.3800,  32.0100,  ...,  99.6500,  10.1500, -64.1500],
        [ 91.0100, -31.3300,  24.3300,  ...,  76.2100, -62.7300,  21.3000],
        [ 66.2900, -52.7500, -61.1200,  ..., -74.4200, -79.0300, -61.4000]])

In [452]:
a = torch.tensor([0.3, 0.3, 0.3, 0.3,
                  -0.3, -0.3, -0.3, -0.3, -0.3, -0.3])
b = torch.tensor([0.95, 0.95, 0.95, 0.95,
                  -0.95, -0.95, -0.95, -0.95, -0.95, -0.95])

train_data, val_test_data = train_test_split(Data, test_size=0.2, shuffle=True)
val_data, test_data = train_test_split(Data, test_size=0.2)

train_dataset = SimpleDataset(train_data, a, b)
val_dataset   = SimpleDataset(val_data, a, b)
test_dataset  = SimpleDataset(test_data, a, b)

train_dataset[0][0], train_dataset.show_input()[0]

(tensor([  2.8610, -22.4620,  20.8250, -15.1120, -24.3830,  -9.7460, -22.3850,
         -22.2080,  -3.6740, -28.9490]),
 tensor([  6.3700, -78.0400,  66.2500, -53.5400,  78.1100,  29.3200,  71.4500,
          70.8600,   9.0800,  93.3300]))

In [453]:
def train_model(model, train_loader, val_data, epochs=30, lr=1e-3, patience=10, min_delta=1e-6):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    val_labels, val_targets = val_data
    best_val_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for targets, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(targets)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        with torch.no_grad():
            # Прямой проход на валидации
            val_outputs_fwd = model(val_targets)
            val_loss_forw = criterion(val_outputs_fwd, val_labels)

            # Обратный проход на валидации (очистка шума)
            val_outputs_rev = model(val_labels, reverse=True)
            val_loss_backw = criterion(val_outputs_rev, val_targets)

        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {total_loss/len(train_loader):.6f}, Val Loss Rev: {val_loss_backw.item():.6f}")

        # Early Stopping
        if val_loss_backw < best_val_loss - min_delta:
            best_val_loss = val_loss_backw
            epochs_no_improve = 0
            #torch.save(model.state_dict(), 'best_model.pth')
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best Val Loss Rev: {best_val_loss:.6f}")
            break

In [454]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
model = SimpleLinearINN(input_dim1=4, input_dim2=6)
val_tuple = (val_dataset.labels, val_dataset.targets)
train_model(model, train_loader, val_data=val_tuple, epochs=1000, lr=1e-3, patience=15, min_delta=1e-10)

Epoch 1/1000, Train Loss: 30.600686, Val Loss Rev: 2.877822
Early stopping at epoch 21. Best Val Loss Rev: 0.010034


In [455]:
test_cleaned = model(test_dataset.labels, reverse=True)

mse_loss = F.mse_loss(test_cleaned, test_dataset.targets)
mae_loss = F.l1_loss(test_cleaned, test_dataset.targets)

targets_mean_sq = torch.mean(test_dataset.targets**2)
relative_mse = mse_loss / (targets_mean_sq + 1e-8)

mape = torch.mean(torch.abs((test_dataset.targets - test_cleaned) / (test_dataset.targets + 1e-8)))

print(f"Absolute MSE: {mse_loss.item():.6f}")
print(f"Absolute MAE: {mae_loss.item():.6f}")
print(f"Relative MSE: {relative_mse.item():.6%}")
print(f"MAPE (Mean Absolute % Error): {mape.item():.6%}")

Absolute MSE: 0.010619
Absolute MAE: 0.082223
Relative MSE: 0.003508%
MAPE (Mean Absolute % Error): 2.837143%


In [456]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

X_train = train_dataset.labels.numpy()
y_train = train_dataset.targets.numpy()
X_test = test_dataset.labels.numpy()
y_test = test_dataset.targets.numpy()

sku_model = LinearRegression()
sku_model.fit(X_train, y_train)

y_pred = sku_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rel_mse = mse / (np.mean(y_test**2) + 1e-8)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8)))

print(f"--- Sklearn Linear Regression Metrics ---")
print(f"Absolute MSE: {mse:.6f}")
print(f"Absolute MAE: {mae:.6f}")
print(f"Relative MSE: {rel_mse:.6%}")
print(f"MAPE: {mape:.6%}")

--- Sklearn Linear Regression Metrics ---
Absolute MSE: 0.010006
Absolute MAE: 0.079685
Relative MSE: 0.003306%
MAPE: 2.723078%
